# 🗄️ Definição do Esquema Relacional do E-commerce (SQLAlchemy & SQLite)

Este notebook define formalmente o esquema relacional DDL (*Data Definition Language*) para o banco de dados **`ecommerce.db`** utilizando **SQLAlchemy Core**.

Ele estabelece a estrutura de tabelas, chaves primárias, chaves estrangeiras e tipos de dados necessários para suportar as consultas SQL analíticas executadas pelos agentes autônomos da **Crew de Marketing Digital**.

---

### 📑 Tabelas do Esquema:
- **`customers`**: Base cadastral de clientes e métricas históricas de valor (LTV, AOV, intervalo de recompra).
- **`sku_catalog`**: Catálogo de produtos, categorias, fornecedores e custos de produção.
- **`meta_ads_campaigns`**: Métricas de anúncios (impressões, cliques, CTR, ROAS, CAC, hook rate).
- **`website_daily`**: Consolidação diária de tráfego, funil e conversão por canal e dispositivo.
- **`website_sessions`**: Registro granular de sessões de navegação e eventos de funil.
- **`orders`**: Registro de pedidos de venda, valores brutos/líquidos, descontos e canais de atribuição.
- **`order_line_items`**: Itens individuais vendidos por pedido, tamanhos, descontos e devoluções.
- **`inventory_snapshots`**: Histórico diário de estoque, projeções de esgotamento e identificação de *dead stock*.
- **`purchase_orders`**: Pedidos de reposição com fornecedores e monitoramento de *lead time*.

## 1. Conexão e Inicialização dos Metadados

Configuração da conexão com o banco SQLite local (`ecommerce.db`) e instanciação do objeto `MetaData` que registrará todas as tabelas do sistema.

In [ ]:
from sqlalchemy import (
    Column,
    Date,
    Float,
    ForeignKey,
    Integer,
    MetaData,
    String,
    Table,
    Time,
    create_engine,
)

# Criação da engine de conexão para o banco SQLite local
engine = create_engine("sqlite:///ecommerce.db")
metadata = MetaData()

## 2. Definição das Tabelas do Esquema Relacional

### 2.1 Tabela `customers` (Clientes)
Armazena o perfil do consumidor, data da primeira compra, valor total gasto e intervalo até a recompra.

In [ ]:
customers_table = Table(
    "customers",
    metadata,
    Column("customer_id", String, primary_key=True, doc="Identificador único do cliente"),
    Column("first_order_date", Date, doc="Data da primeira compra"),
    Column("total_orders", Integer, doc="Quantidade total de pedidos realizados"),
    Column("total_revenue", Float, doc="Receita total gerada pelo cliente"),
    Column("average_order_value", Float, doc="Ticket médio do cliente (AOV)"),
    Column("time_to_2nd_purchase", Float, doc="Dias decorridos até a segunda compra"),
    Column("last_purchase_date", Date, doc="Data da última compra realizada"),
    Column("city_tier", String, doc="Classificação demográfica da cidade do cliente"),
    Column("acquisition_channel", String, doc="Canal de marketing que originou o cliente"),
    Column("repurchased", String, doc="Indicador se o cliente efetuou recompra (Yes/No)"),
)

### 2.2 Tabela `sku_catalog` (Catálogo de SKUs)
Contém os produtos disponíveis, preços de varejo recomendados (MRP) e custos de fabricação/aquisição.

In [ ]:
sku_catalog_table = Table(
    "sku_catalog",
    metadata,
    Column("sku", String, primary_key=True, doc="Código único do SKU"),
    Column("category", String, doc="Categoria do produto"),
    Column("vendor", String, doc="Fornecedor responsável pela confecção"),
    Column("mrp", Integer, doc="Preço máximo de varejo tabelado"),
    Column("cost_per_unit", Integer, doc="Custo unitário de produção do item"),
)

### 2.3 Tabela `meta_ads_campaigns` (Campanhas de Anúncios)
Histórico de veiculação no Meta Ads: investimento, alcance, cliques, adições ao carrinho e retorno (ROAS).

In [ ]:
meta_ads_campaigns_table = Table(
    "meta_ads_campaigns",
    metadata,
    Column("adset_name", String, primary_key=True, doc="Nome do conjunto de anúncios"),
    Column("results", Integer, doc="Resultados de conversão obtidos"),
    Column("spend", Integer, doc="Investimento realizado na campanha"),
    Column("reach", Integer, doc="Pessoas únicas alcançadas"),
    Column("impressions", Integer, doc="Total de impressões exibidas"),
    Column("frequency", Float, doc="Frequência média de exibição"),
    Column("link_clicks", Integer, doc="Cliques no link do anúncio"),
    Column("ctr", Float, doc="Taxa de cliques (Click-Through Rate)"),
    Column("add_to_cart", Integer, doc="Adições ao carrinho originadas pelo anúncio"),
    Column("initiate_checkout", Integer, doc="Checkouts iniciados"),
    Column("purchases", Integer, doc="Compras concluídas"),
    Column("conversion_value", Float, doc="Valor total das conversões (receita atribuída)"),
    Column("cac", Float, doc="Custo de Aquisição de Clientes (CAC)"),
    Column("roi", Float, doc="Retorno sobre o Investimento (ROAS / ROI)"),
    Column("creative_type", String, doc="Formato do criativo (estático, vídeo, carrossel)"),
    Column("launch_date", Date, doc="Data de lançamento do anúncio"),
    Column("hook_rate", Float, doc="Taxa de retenção nos primeiros segundos do vídeo"),
)

### 2.4 Tabela `website_daily` (Métricas Diárias de Tráfego)
Consolidação por dia, canal e dispositivo, permitindo analisar taxas de abandono e conversão de topo/meio de funil.

In [ ]:
website_daily_table = Table(
    "website_daily",
    metadata,
    Column("daily_date", Date, doc="Data da agregação diária"),
    Column("traffic_source", String, doc="Origem do tráfego (Meta, Google, Orgânico, etc.)"),
    Column("campaign_name", String, doc="Nome da campanha UTM associada"),
    Column("device_category", String, doc="Dispositivo utilizado (mobile, desktop, tablet)"),
    Column("daily_sessions", Integer, doc="Total de sessões registradas no dia"),
    Column("daily_product_views", Integer, doc="Visualizações de páginas de produto"),
    Column("daily_add_to_cart", Integer, doc="Eventos de adição ao carrinho no dia"),
    Column("daily_begin_checkout", Integer, doc="Eventos de início de checkout no dia"),
    Column("daily_purchases", Integer, doc="Transações concluídas"),
    Column("daily_revenue", Float, doc="Faturamento diário total"),
    Column("daily_conversion_rate", Float, doc="Taxa de conversão diária do tráfego"),
    Column("daily_aov", Float, doc="Ticket médio diário (Average Order Value)"),
    Column("country", String, doc="País de navegação"),
    Column("city_location", String, doc="Cidade do usuário"),
    Column("has_purchases_flag", Integer, doc="Flag indicando se houve vendas no registro"),
)

### 2.5 Tabela `website_sessions` (Sessões Granulares)
Contém dados individuais de cada sessão de navegação, mapeando a jornada desde a visita até a compra.

In [ ]:
website_sessions_table = Table(
    "website_sessions",
    metadata,
    Column("session_id", String, primary_key=True, doc="Identificador único da sessão"),
    Column("session_datetime", String, doc="Data e hora de início da sessão"),
    Column("session_source", String, doc="Canal de origem da sessão"),
    Column("campaign_name", String, doc="Campanha de marketing de origem"),
    Column("user_device", String, doc="Dispositivo utilizado pelo usuário"),
    Column("user_city", String, doc="Cidade do usuário"),
    Column("session_count", Integer, doc="Contagem de acessos na sessão"),
    Column("session_product_views", Integer, doc="Visualizações de produto na sessão"),
    Column("session_cart_additions", Integer, doc="Produtos adicionados ao carrinho na sessão"),
    Column("session_checkouts_started", Integer, doc="Checkouts iniciados pelo visitante"),
    Column("is_purchase_sucessful", Integer, doc="Flag indicando se a sessão resultou em compra"),
    Column("order_id", String, doc="ID do pedido associado (se houver)"),
    Column("customer_id", String, doc="ID do cliente associado à sessão"),
    Column("session_revenue", Float, doc="Receita monetária atribuída à sessão"),
)

### 2.6 Tabela `orders` (Pedidos de Venda)
Registra cada pedido concluído, incluindo forma de pagamento, valores brutos e líquidos e canal de conversão.

In [ ]:
orders_table = Table(
    "orders",
    metadata,
    Column("order_id", String, primary_key=True, doc="Identificador único do pedido"),
    Column("customer_id", String, ForeignKey("customers.customer_id"), doc="Chave estrangeira do cliente"),
    Column("product", String, doc="Descrição resumida do produto ou pacote"),
    Column("gross_value", Integer, doc="Valor bruto total do pedido"),
    Column("net_value", Integer, doc="Valor líquido faturado após descontos"),
    Column("discount_value", Integer, doc="Valor total do desconto concedido"),
    Column("discount_percentage", Float, doc="Percentual de desconto aplicado"),
    Column("payment_mode", String, doc="Método de pagamento (Cartão, PIX, etc.)"),
    Column("shipping_city", String, doc="Cidade de entrega do pedido"),
    Column("first_order_vs_repeat", String, doc="Classificação (primeira compra vs recompra)"),
    Column("last_touch_channel", String, doc="Canal de último toque na jornada"),
    Column("order_status", String, doc="Status do pedido (Delivered, Shipped, Cancelled)"),
    Column("order_date", Date, doc="Data do pedido"),
    Column("order_time", Time, doc="Horário da transação"),
)

### 2.7 Tabela `order_line_items` (Itens do Pedido)
Detalha cada SKU vendido em cada pedido, possibilitando análise de devoluções e categorias populares.

In [ ]:
order_line_items_table = Table(
    "order_line_items",
    metadata,
    Column("order_id", String, ForeignKey("orders.order_id"), doc="Chave estrangeira do pedido"),
    Column("sku", String, ForeignKey("sku_catalog.sku"), doc="Chave estrangeira do SKU"),
    Column("category", String, doc="Categoria de vestuário/produto"),
    Column("size", String, doc="Tamanho da peça (S, M, L, XL)"),
    Column("color", String, doc="Cor do item"),
    Column("mrp", Integer, doc="Preço de tabela"),
    Column("selling_price", Integer, doc="Preço de venda praticado"),
    Column("discount_percentage", Float, doc="Percentual de desconto específico do item"),
    Column("returned", String, doc="Indicador de devolução (Yes/No)"),
    Column("return_reason", String, doc="Motivo registrado para a devolução"),
)

### 2.8 Tabela `inventory_snapshots` (Posição de Estoque)
Rastreamento de níveis de estoque, velocidade de vendas (7d, 30d, 60d) e estoque parado (*dead stock*).

In [ ]:
inventory_snapshots_table = Table(
    "inventory_snapshots",
    metadata,
    Column("sku", String, ForeignKey("sku_catalog.sku"), doc="Chave estrangeira do SKU"),
    Column("category", String, doc="Categoria do produto em estoque"),
    Column("size", String, doc="Tamanho do item"),
    Column("units_in_stock", Integer, doc="Unidades físicas disponíveis no momento"),
    Column("units_sold_7d", Integer, doc="Unidades vendidas nos últimos 7 dias"),
    Column("units_sold_30d", Integer, doc="Unidades vendidas nos últimos 30 dias"),
    Column("units_sold_60d", Integer, doc="Unidades vendidas nos últimos 60 dias"),
    Column("days_of_inventory_left", Integer, doc="Projeção de dias até o esgotamento do estoque"),
    Column("dead_stock_flag", String, doc="Sinalizador de estoque encalhado (Yes/No)"),
    Column("date", Date, doc="Data do registro do snapshot"),
)

### 2.9 Tabela `purchase_orders` (Ordens de Compra de Reposição)
Monitora as solicitações de reposição junto aos fornecedores e o *lead time* até a entrega física.

In [ ]:
purchase_orders_table = Table(
    "purchase_orders",
    metadata,
    Column("sku", String, ForeignKey("sku_catalog.sku"), doc="Chave estrangeira do SKU solicitado"),
    Column("vendor", String, doc="Fornecedor responsável pelo pedido"),
    Column("order_quantity", Integer, doc="Quantidade de unidades encomendadas"),
    Column("cost_per_unit", Integer, doc="Custo unitário de compra contratado"),
    Column("order_date", Date, doc="Data de emissão da ordem de compra"),
    Column("expected_delivery", Date, doc="Data prevista para entrega"),
    Column("actual_delivery", Date, doc="Data real da entrega"),
    Column("lead_time", Integer, doc="Tempo total de espera em dias até o recebimento"),
)

## 3. Criação das Tabelas no Banco SQLite

Executa a criação de todas as tabelas registradas no `metadata` caso ainda não existam no arquivo `ecommerce.db`.

In [ ]:
# Criação de todas as tabelas no arquivo SQLite local
metadata.create_all(engine)
print("✅ Banco de dados ecommerce.db e todas as tabelas verificadas/criadas com sucesso!")

# Inspeção e validação das tabelas criadas
from sqlalchemy import inspect
inspector = inspect(engine)
tables = inspector.get_table_names()
print(f"\n📋 Tabelas registradas no banco ({len(tables)}):")
for t in sorted(tables):
    cols = [c['name'] for c in inspector.get_columns(t)]
    print(f"  - {t}: {len(cols)} colunas")